# Laboratory 9: BB84 on the Bench — Analyzing Your Data

At the bench you ran BB84 with the Thorlabs EDU-QCRY1 kit: dice chose every basis and every
bit, the laser and the receivers did the rest, and your worksheet recorded it all. This
notebook turns worksheets into numbers, using only numpy, pandas and matplotlib (one
optional cell at the very end uses Qiskit). The questions driving the five parts:

1. **The manual's example** — the kit manual runs BB84 by hand on 18 pulses, once without
   Eve and once with her. Does its bookkeeping add up?
2. **One worksheet** — how does a column of `same` / `diff` marks become an error rate,
   and how far can one group's number be trusted?
3. **The class** — six groups faced the same Eve and got six different error rates. Why
   pool them?
4. **The 6-bit test** — would 6 test bits have caught Eve?
5. **Your own bits** — are you a good random number generator?

> **SYNTHETIC data.** Until real runs from the bench exist, Parts 2–5 run on a
> **SYNTHETIC** class data set (6 groups), drawn from the ideal bench model with seed 909.
> It is **not** a real bench run: replace it with your class's worksheets (Part 2 shows how).

**The ideal bench.** Every prediction here, and the synthetic data, assume an ideal bench
(declared deviation DEV-9.3): perfect plates and sensors, no light lost, no noise. A real
bench is slightly misaligned and can show a few errors even without Eve.

**Conventions.** States follow the course convention $|0\rangle \equiv |H\rangle$
(horizontal), $|1\rangle \equiv |V\rangle$, $|+\rangle$ at $+45°$ and $|-\rangle$ at
$-45°$. Bases and bits use the kit's labels, exactly as on the worksheet: bases `+` and
`x`, bits `0` and `1`, where a receiver records transmitted = 0 and reflected = 1. In the
`x` basis, bit 0 is the $-45$ setting on Alice's dial and bit 1 the $45$ setting.

In [ ]:
"""
Setup: libraries, the seed, the data loader and the worksheet parser.
Run this cell first. No quantum library is needed until the optional last cell.
"""
import io
import math
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

SEED = 909      # the seed of the SYNTHETIC data set; the optional Qiskit cell reuses it
RAW_BASE = "https://raw.githubusercontent.com/Quantum-Plus-Labs/quantum-labs-2026/main/lab9/"
LOADED_FROM = {}    # file name -> where it was read from

DATA_COLOR, PREDICTION_COLOR = "#2a78d6", "#eb6834"    # charts: blue = data, orange = prediction


def load_csv(name):
    """Read a lab CSV: the file next to this notebook if present, else the course repository.

    Lines starting with '#' carry the source and provenance: they are returned as notes and
    skipped by pandas (comment='#'). Every value is read as text, so a blank cell stays "".
    """
    local = Path(name)
    if local.is_file():
        text, source = local.read_text(encoding="utf-8"), str(local)
    else:
        with urllib.request.urlopen(RAW_BASE + name) as response:
            text, source = response.read().decode("utf-8"), RAW_BASE + name
    LOADED_FROM[name] = source
    notes = [line.rstrip().removeprefix("# ").removeprefix("#")
             for line in text.splitlines() if line.startswith("#")]
    table = pd.read_csv(io.StringIO(text), comment="#", dtype=str, keep_default_na=False)
    return table, notes


WORKSHEET_COLUMNS = ["group", "run", "pulse", "eve_basis", "alice_basis", "alice_bit",
                     "bob_basis", "bob_bit", "test"]
ALLOWED = {"run": ("1", "2"), "alice_basis": ("+", "x"), "bob_basis": ("+", "x"),
           "eve_basis": ("+", "x", ""), "alice_bit": ("0", "1"), "bob_bit": ("0", "1")}
# The test column: same / diff for a publicly compared bit, blank = not tested.
# The worksheet's signs are accepted too: = for same, ≠ or != for diff.
TEST_MARKS = {"same": "same", "=": "same", "diff": "diff", "≠": "diff", "!=": "diff", "": ""}


def parse_worksheet(table):
    """Check every code of one or more worksheets and add the analysis columns.

    Rows of pulses that were not fired (bases and bits all blank) are dropped. A bad code
    stops with the row to fix. A test mark that disagrees with the bits, or sits on a pulse
    whose bases differ, is FLAGGED and listed, never silently fixed: errors are always
    counted from the bits. Returns (pulses, flagged).
    """
    missing = [c for c in WORKSHEET_COLUMNS if c not in table.columns]
    if missing:
        raise ValueError(f"missing columns: {missing}")
    df = table[WORKSHEET_COLUMNS].apply(lambda column: column.astype(str).str.strip())
    fired = (df[["alice_basis", "alice_bit", "bob_basis", "bob_bit"]] != "").any(axis=1)
    df = df[fired].reset_index(drop=True)
    df["test"] = df["test"].str.lower()
    for column, allowed in {**ALLOWED, "test": tuple(TEST_MARKS)}.items():
        bad = df[~df[column].isin(allowed)]
        if len(bad):
            row = bad.iloc[0]
            raise ValueError(f"{column} = {row[column]!r} (group {row['group']}, run {row['run']}, "
                             f"pulse {row['pulse']}); allowed: {allowed}")
    for column in ("run", "pulse", "alice_bit", "bob_bit"):
        df[column] = df[column].astype(int)
    df["mark"] = df["test"].map(TEST_MARKS)
    df["sifted"] = df["alice_basis"] == df["bob_basis"]
    df["bits_differ"] = df["alice_bit"] != df["bob_bit"]
    tested = df["mark"] != ""
    df["test_bit"] = tested & df["sifted"]
    off_sift = tested & ~df["sifted"]
    disagree = df["test_bit"] & (df["mark"] != np.where(df["bits_differ"], "diff", "same"))
    flagged = df[off_sift | disagree].copy()
    flagged["reason"] = np.where(off_sift[off_sift | disagree], "mark on a pulse whose bases differ",
                                 "mark disagrees with the bits")
    return df, flagged


print("Setup complete. Seed of the SYNTHETIC data:", SEED)

---
## 1 · Does the Manual's Own Example Add Up?

**The question first:** *the kit manual (Chapter 6) runs BB84 by hand on 18 pulses, once
without Eve and once with her. Redo its bookkeeping from the raw per-pulse tables: do you
get its key, its ciphertext and its errors?*

**Why this step:** those raw tables are exactly what your worksheet holds, so they are
the perfect rehearsal. Three ideas, each checked below:

- **Sifting** keeps only the pulses where Alice's and Bob's bases agree; bases are
  compared in public, bits never. On a matched pulse an ideal bench hands Bob Alice's
  bit ($b = a$); on a mismatched one his bit is a coin flip. The sifted bits are the key.
- **The one-time pad** encrypts a message bit $m$ with a key bit $k$ by XOR,
  $y = m \oplus k$ (with $0\oplus0 = 1\oplus1 = 0$ and $0\oplus1 = 1\oplus0 = 1$). Since
  $k \oplus k = 0$, Bob recovers $(m \oplus k) \oplus k = m$. The key must be as long as
  the message, random, and used only once. Letters use the kit's 5-bit alphabet: the
  letter's position from A = 0 to Z = 25, in binary (A = 00000, …, Z = 11001).
- **Eve's footprint.** Eve does *intercept-resend*: she measures each pulse in a basis of
  her own and re-sends what she read. Where her basis equals Alice's she re-sends Alice's
  bit exactly; only where it differs can Bob's bit come out wrong.

The data file `lab9_manual_ch6.csv` holds the manual's tables transcribed per pulse, with
attribution (printed below); everything derived from them is recomputed here.

In [ ]:
"""
Part 1 - the manual's Chapter 6 example, recomputed from its per-pulse tables.
"""
ch6, notes = load_csv("lab9_manual_ch6.csv")
print("\n".join(notes), "\n")
for column in ("pulse", "alice_bit", "bob_bit_no_eve", "eve_bit", "bob_bit_with_eve"):
    ch6[column] = ch6[column].astype(int)
assert list(ch6["pulse"]) == list(range(1, 19))

# Sifting: keep the pulses where Alice's and Bob's bases agree.
same_basis = ch6["alice_basis"] == ch6["bob_basis"]
sifted = ch6.loc[same_basis, "pulse"].tolist()
print("same-basis pulses:", sifted)

# Section 6.1, no Eve: Bob's sifted bits equal Alice's, and that string is the key.
key = "".join(map(str, ch6.loc[same_basis, "bob_bit_no_eve"]))
assert key == "".join(map(str, ch6.loc[same_basis, "alice_bit"]))    # b = a on matched bases
assert key == "0110010001"
print("sifted key (Bob's bits = Alice's bits):", key)


def letter_bits(letter):
    """The kit's 5-bit code: the letter's position, A = 0 ... Z = 25, in binary."""
    return format(ord(letter) - ord("A"), "05b")


def xor(u, v):
    """Bitwise XOR of two equally long bit strings."""
    assert len(u) == len(v), "the key must be exactly as long as the message"
    return "".join(str(int(p) ^ int(q)) for p, q in zip(u, v))


assert letter_bits("A") == "00000" and letter_bits("Z") == "11001"
message = letter_bits("Q") + letter_bits("M")
ciphertext = xor(message, key)
print(f"'QM' = {message}  XOR  key {key}  =  ciphertext {ciphertext}")
assert message == "1000001100" and ciphertext == "1110011101"
assert xor(ciphertext, key) == message                    # (m XOR k) XOR k = m: Bob decrypts

# Section 6.2, with Eve: Bob's sifted bits, his errors, and where Eve's basis was not Alice's.
bob_sifted_6_2 = "".join(map(str, ch6.loc[same_basis, "bob_bit_with_eve"]))
wrong = same_basis & (ch6["bob_bit_with_eve"] != ch6["alice_bit"])
eve_other_basis = same_basis & (ch6["eve_basis"] != ch6["alice_basis"])
error_pulses = ch6.loc[wrong, "pulse"].tolist()
eve_differs_pulses = ch6.loc[eve_other_basis, "pulse"].tolist()
print(f"\nwith Eve: Bob's sifted bits {bob_sifted_6_2}, errors at pulses {error_pulses} "
      f"({len(error_pulses)} of {len(sifted)})")
print(f"          Eve's basis differs from Alice's at the sifted pulses {eve_differs_pulses}")
assert bob_sifted_6_2 == "0100000000" and error_pulses == [5, 8, 18]
assert set(error_pulses) <= set(eve_differs_pulses)     # every error sits where Eve's basis was wrong
print(f"every error sits where Eve's basis differed from Alice's: {len(error_pulses)} of those "
      f"{len(eve_differs_pulses)} pulses erred, {len(eve_differs_pulses) / 2:g} on average "
      "(each errs with probability 1/2)")

---
## 2 · From a Worksheet to a QBER

**The question first:** *your run-2 worksheet carries a `same` / `diff` mark on every
same-basis pulse. Which single number says whether Eve was on the line, and how much can
one group's value of it be trusted?*

**Why this step:** the **QBER** (quantum bit error rate) is the probability that Bob's
sifted bit differs from Alice's. Its estimate from $t$ test bits is

$$\hat Q = \frac{\text{errors}}{t}.$$

Without Eve, on an ideal bench, $Q = 0$. With intercept-resend Eve on every pulse, her
basis matches Alice's on half the sifted pulses (no error) and misses on the other half,
where Bob's bit becomes a coin flip (error $\tfrac12$), so
$Q = \tfrac12\cdot 0 + \tfrac12\cdot\tfrac12 = \tfrac14 = 25\,\%$. The **standard error**
(SE) is the typical size of the random scatter of an estimate around the true value; for
$t$ test bits

$$\text{SE} = \sqrt{\frac{Q(1-Q)}{t}}.$$

As on the worksheet (W4), the SE is evaluated at $Q = 1/4$: it says how precisely this
many test bits can tell 0 from 25 %.

By default the cell analyzes **group G1, run 2, of the SYNTHETIC class data (seed 909)**.

**Your own data.** Copy `lab9_worksheet_template.csv` (in the course repository, next to
this notebook), fill one row per pulse from your worksheet (its `#` lines explain the
codes; type `x` for the worksheet's ×) and save it under a new name. Put it next to this notebook (in Colab: the Files
panel, then Upload), set `CLASS_FILE` to its name and `MY_GROUP` to your group's name,
and rerun from this cell on. A class file is simply every group's rows in one CSV.

In [ ]:
"""
Part 2 - one worksheet to a QBER and its standard error.
"""
SYNTHETIC_FILE = "lab9_synthetic_class.csv"
CLASS_FILE = SYNTHETIC_FILE     # <- your class's CSV (same columns as lab9_worksheet_template.csv)
MY_GROUP = "G1"                 # <- your group's name in that file
USING_SYNTHETIC = CLASS_FILE == SYNTHETIC_FILE

raw, notes = load_csv(CLASS_FILE)
print("\n".join(notes), "\n")
pulses, flagged = parse_worksheet(raw)
if len(flagged):
    print(f"FLAGGED: {len(flagged)} test mark(s) to check on the worksheet "
          "(errors are counted from the bits, the marks are left as written):")
    print(flagged[["group", "run", "pulse", "alice_bit", "bob_bit", "test", "reason"]].to_string(index=False))
else:
    print("every test mark agrees with the bits and sits on a same-basis pulse")

Q_EVE = 0.5 * 0.0 + 0.5 * 0.5   # Eve's basis right (1/2): no error; wrong (1/2): error 1/2
assert Q_EVE == 0.25


def standard_error(t, q=Q_EVE):
    """SE of an error rate from t test bits, sqrt(Q(1 - Q)/t), at Q = 1/4 as on the worksheet."""
    return math.sqrt(q * (1 - q) / t) if t else math.nan


def run_counts(run_pulses):
    """N pulses, S sifted, t test bits, errors, QBER = errors/t, and SE at Q = 1/4."""
    t = int(run_pulses["test_bit"].sum())
    errors = int((run_pulses["test_bit"] & run_pulses["bits_differ"]).sum())
    return {"N": len(run_pulses), "S": int(run_pulses["sifted"].sum()), "t": t,
            "errors": errors, "QBER": errors / t if t else math.nan, "SE": standard_error(t)}


mine = pulses[(pulses["group"] == MY_GROUP) & (pulses["run"] == 2)]
tested = mine[mine["test_bit"]]
print(f"\n{MY_GROUP}, run 2: the {len(tested)} tested same-basis pulses, in order")
print("  Alice's bits  ", "".join(map(str, tested["alice_bit"])))
print("  Bob's bits    ", "".join(map(str, tested["bob_bit"])))
print("  errors        ", "".join("^" if d else " " for d in tested["bits_differ"]))

my_run2 = run_counts(mine)
print(f"\nN = {my_run2['N']} pulses, S = {my_run2['S']} same-basis, t = {my_run2['t']} tested, "
      f"errors = {my_run2['errors']}")
print(f"QBER = {my_run2['errors']}/{my_run2['t']} = {my_run2['QBER']:.4f}")
print(f"SE at Q = 1/4 = {my_run2['SE']:.4f}  (how precisely {my_run2['t']} test bits can tell 0 from 25 %)")
eve_right_errors = int((tested["bits_differ"] & (tested["eve_basis"] == tested["alice_basis"])).sum())
print("errors where Eve's basis matched Alice's:", eve_right_errors, "(none possible on the ideal bench)")
assert my_run2["t"] == 0 or my_run2["QBER"] == my_run2["errors"] / my_run2["t"]
if USING_SYNTHETIC:
    assert len(flagged) == 0 and eve_right_errors == 0

---
## 3 · Why Pool the Class Data?

**The question first:** *six groups faced the same Eve and measured six different
QBERs. Is any of them wrong, and what is the class's best answer?*

**Why this step:** none of them is wrong. With about 26 test bits a group's SE at
$Q = 1/4$ is about 0.085 (how precisely 26 test bits can tell 0 from 25 %), so single
groups scatter by several percentage points either way. Pooling adds everyone's counts:

$$\hat Q_{\text{class}} = \frac{\sum \text{errors}}{\sum t}, \qquad
  \text{SE}_{\text{class}} = \sqrt{\frac{Q(1-Q)}{\sum t}} \quad \text{at } Q = \tfrac14 ,$$

so $G$ groups with $t$ test bits each shrink the SE by a factor $\sqrt{G}$. Run 1 (no
Eve) must read exactly 0 on the ideal bench: with no Eve and a perfect bench a
same-basis bit cannot flip. On a real bench a few run-1 errors point at misalignment,
not at Eve.

The table covers both runs of every group in the **SYNTHETIC** data (seed 909). Run 1
needs $S \ge 26$ (20 key bits for a 4-letter word plus 6 test bits); a group still short
after its extra pulses gets a note under the table. In the synthetic data that is G1's
run 1, the roughly 1 % case.

In [ ]:
"""
Part 3 - the class table, the pooled QBER and SE per run, and the run-2 plot.
"""
class_table = pd.DataFrame([{"group": group, "run": run, **run_counts(run_pulses)}
                            for (group, run), run_pulses in pulses.groupby(["group", "run"], sort=False)])
print("per group and run (SE at Q = 1/4: how precisely t test bits can tell 0 from 25 %)")
print(class_table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# Run 1 budget: 20 key bits (a 4-letter word) + 6 test bits need S >= 26.
p_s26_70 = sum(math.comb(70, j) for j in range(26, 71)) / 2 ** 70     # P(S >= 26) with 70 pulses
for r in class_table[(class_table["run"] == 1) & (class_table["S"] < 26)].itertuples():
    key_bits, letters = r.S - r.t, (r.S - r.t) // 5
    short = "one letter" if letters == 3 else f"{4 - letters} letters"
    if r.N >= 70:
        print(f"note: {r.group}, run 1: S = {r.S} even after {r.N} pulses (70 pulses reach S >= 26 with "
              f"probability {p_s26_70:.1%}: this is the {1 - p_s26_70:.1%} case). {r.t} test bits leave "
              f"{key_bits} key bits, enough for {letters} letters: {short} short of a 4-letter word. "
              "At the bench: ask the staff.")
    else:
        print(f"note: {r.group}, run 1: S = {r.S} after {r.N} pulses, short of 26: the bench guide "
              "fires extra pulses, up to 70, until S >= 26.")

pooled = class_table.groupby("run", as_index=False)[["errors", "t"]].sum()
pooled["QBER"] = pooled["errors"] / pooled["t"]
pooled["SE"] = [standard_error(t) for t in pooled["t"]]
print("\nclass-pooled (SE at Q = 1/4: how precisely all the class's test bits together can tell 0 from 25 %)")
print(pooled.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

assert abs(standard_error(26) - 0.085) < 5e-4                            # the value quoted above
assert abs(standard_error(6 * 26) * math.sqrt(6) - standard_error(26)) < 1e-12   # G groups: SE / sqrt(G)
p1, p2 = pooled.set_index("run").loc[1], pooled.set_index("run").loc[2]
print(f"\nrun 1: class QBER {p1.QBER:.4f}"
      + (" = 0, as the ideal bench predicts" if p1.errors == 0 else ": not 0, so look at the alignment"))
print(f"run 2: class QBER {p2.QBER:.4f} ± {p2.SE:.4f}, "
      f"{abs(p2.QBER - Q_EVE) / p2.SE:.2f} SE away from the 25 % prediction")
if USING_SYNTHETIC:
    assert p1.errors == 0 and abs(p2.QBER - Q_EVE) < 3 * p2.SE

run2 = class_table[class_table["run"] == 2]
x = np.arange(len(run2))
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.axhspan(p2.QBER - p2.SE, p2.QBER + p2.SE, color=DATA_COLOR, alpha=0.15, lw=0,
           label="class pooled QBER ± SE")
ax.axhline(p2.QBER, color=DATA_COLOR, lw=1)
ax.axhline(Q_EVE, color=PREDICTION_COLOR, lw=2, ls="--", label="intercept-resend prediction, 25 %")
ax.errorbar(x, run2["QBER"], yerr=run2["SE"], fmt="o", ms=8, lw=2, capsize=4, color=DATA_COLOR,
            label="group QBER ± SE")
ax.set_xticks(x)
ax.set_xticklabels(run2["group"])
ax.set_ylim(0, max(0.6, float((run2["QBER"] + run2["SE"]).max()) + 0.05))
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylabel("run-2 QBER")
ax.set_title(("SYNTHETIC data, seed 909" if USING_SYNTHETIC else CLASS_FILE)
             + ": run 2, Eve on every pulse (SE at Q = 1/4)", fontsize=10)
ax.grid(axis="y", color="0.9")
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

---
## 4 · Would 6 Test Bits Have Caught Eve?

**The question first:** *in run 1 you tested only 6 bits before encrypting. Had Eve been
on the line, would those 6 bits have given her away?*

**Why this step:** with Eve on every pulse each test bit shows an error with probability
$Q = 1/4$, independently of the others, so she slips through $t$ test bits unseen with
probability

$$P_{\text{undetected}}(t) = \left(\tfrac34\right)^t .$$

At $t = 6$ that is 17.8 %: a 6-bit test misses Eve about one time in six and catches
her with probability $1 - (3/4)^6 = 82.2\,\%$. Pushing the miss below 1 % takes
$t = 17$. Run 2 lets you watch this happen: read only the **first 6 same-basis bits** of
each group's run 2, as if the test had stopped there, and count the groups that would
have seen at least one error. By default these are the **SYNTHETIC** data (seed 909).
With only 6 groups the observed fraction scatters widely around 82.2 %, so the code
also computes how likely the observed count is.

In [ ]:
"""
Part 4 - P(undetected) = (3/4)^t, and the first 6 same-basis bits of every group's run 2.
"""
P_UNDET_6 = 0.75 ** 6
assert P_UNDET_6 == 729 / 4096                          # exact in floating point
T_MIN = next(t for t in range(1, 100) if 0.75 ** t < 0.01)
assert T_MIN == 17 and 0.75 ** 16 >= 0.01
print(f"P(undetected) with t = 6 test bits: {P_UNDET_6:.1%};  "
      f"first t below 1 %: t = {T_MIN} ({0.75 ** T_MIN:.2%})")

t_axis = np.arange(0, 31)
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(t_axis, 0.75 ** t_axis, color=DATA_COLOR, lw=2)
ax.axhline(0.01, color="0.55", lw=1, ls=":")
ax.annotate("1 %", (30, 0.01), xytext=(0, 4), textcoords="offset points", ha="right", color="0.35")
for t, label in ((6, f"t = 6: {P_UNDET_6:.1%}"), (T_MIN, f"t = {T_MIN}: {0.75 ** T_MIN:.2%}")):
    ax.plot(t, 0.75 ** t, "o", ms=8, color=DATA_COLOR, markeredgecolor="white", markeredgewidth=2)
    ax.annotate(label, (t, 0.75 ** t), xytext=(8, 8), textcoords="offset points")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("test bits t")
ax.set_ylabel("P(Eve undetected)")
ax.set_title(r"$P_{\mathrm{undetected}}(t) = (3/4)^t$, Eve on every pulse", fontsize=10)
ax.grid(axis="y", color="0.9")
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

FIRST_T = 6
caught = pd.Series({group: bool(run_pulses[run_pulses["sifted"]].head(FIRST_T)["bits_differ"].any())
                    for group, run_pulses in pulses[pulses["run"] == 2].groupby("group", sort=False)},
                   name="caught")
k, G = int(caught.sum()), len(caught)
caught_fraction = k / G
p_catch = 1 - P_UNDET_6                                   # one group's chance to catch Eve
p_count = math.comb(G, k) * p_catch ** k * (1 - p_catch) ** (G - k)
print("\nan error among the first 6 same-basis bits of run 2:",
      ", ".join(f"{g} {'yes' if c else 'no'}" for g, c in caught.items()))
print(f"observed: {k}/{G} = {caught_fraction:.1%} of the groups;  expected: 1 - (3/4)^6 = {p_catch:.1%}")
print(f"P(exactly {k} of {G} groups catch Eve) = {p_count:.1%}")
print(f"With {G} group{'s' if G > 1 else ''}, {k}/{G} is " + ("not surprising." if p_count >= 0.05 else "unusual: check the worksheets."))
if USING_SYNTHETIC:
    assert k == G == 6 and abs(p_count - p_catch ** 6) < 1e-15

---
## 5 · Are You a Good Random Number Generator?

**The question first:** *at the bench the dice chose every basis and every bit. Could
your head have done the dice's job?*

**Why this step:** the pre-lab page's playbox asked you to write 52 bits as if you were
the dice. A **run** of 1s is a block of consecutive 1s with a 0, or the end of the
string, on each side: `0111010` has two runs, of length 3 and 1. For a fair source
(every bit 0 or 1 with probability ½, independently of the others), a run of exactly $n$
ones in the middle of the string needs $n + 2$ bits to come out right (a 0, $n$ ones, a
0), so every extra 1 halves its chance. Counting every place a run can start (at the two
ends of the string one fewer bit has to match), the expected number of runs of length
exactly $n < L$ in $L$ fair bits is

$$E[N_n] = \frac{L - n + 3}{2^{\,n+2}},$$

$2^{-L}$ for $n = L$, and $(L + 1)/4$ runs in total. As $L$ grows, the fraction of runs
that have length $n$ tends to $2^{-n}$: half of all runs are single 1s, a quarter are
pairs, and so on. This is the same law as the playbox. People asked to write random bits
rarely write long runs, because long runs do not look random.

Paste your playbox bits between the triple quotes of `MY_BITS` (spaces and line breaks are ignored). Left empty,
the cell uses the class's dice bits instead: Alice's bit column of the class file, in
file order (group, run, pulse), by default the **SYNTHETIC** data (seed 909). Alice's
8-sided die makes her bit a fair one.

In [ ]:
"""
Part 5 - the run-length test on your bits (or on the class's dice bits).
"""
MY_BITS = """
"""   # <- paste your playbox bits between the triple quotes (line breaks are fine), e.g. 0110 1001 1101


def parse_bits(text):
    """'0 1 1\n0' -> [0, 1, 1, 0]; whitespace is ignored, any other character is an error."""
    bits = []
    for position, ch in enumerate(text, start=1):
        if ch in "01":
            bits.append(int(ch))
        elif not ch.isspace():
            raise ValueError(f'not a bit: "{ch}" (character {position})')
    return bits


def one_run_lengths(bits):
    """Lengths of the maximal runs of 1s, left to right."""
    runs, current = [], 0
    for bit in bits:
        if bit:
            current += 1
        elif current:
            runs.append(current)
            current = 0
    if current:
        runs.append(current)
    return runs


def expected_run_count(L, n):
    """Expected number of 1-runs of length exactly n in L fair bits."""
    if n > L:
        return 0.0
    if n == L:
        return 2.0 ** -L
    return (L - n + 3) / 2 ** (n + 2)


def expected_total_runs(L):
    """Expected number of 1-runs in L fair bits: (L + 1)/4."""
    return (L + 1) / 4


# The law, checked against every one of the 2^12 strings of 12 bits:
L_CHECK = 12
tally = np.zeros(L_CHECK + 1)
for word in range(2 ** L_CHECK):
    for n in one_run_lengths([(word >> (L_CHECK - 1 - j)) & 1 for j in range(L_CHECK)]):
        tally[n] += 1
assert all(abs(tally[n] / 2 ** L_CHECK - expected_run_count(L_CHECK, n)) < 1e-12 for n in range(1, L_CHECK + 1))
assert abs(tally.sum() / 2 ** L_CHECK - expected_total_runs(L_CHECK)) < 1e-12

if MY_BITS.strip():
    run_bits, source = parse_bits(MY_BITS), "your playbox bits"
else:
    run_bits = pulses["alice_bit"].tolist()
    source = "the class's dice bits (Alice's bit column)" + (", SYNTHETIC, seed 909" if USING_SYNTHETIC else "")
runs = one_run_lengths(run_bits)
L = len(run_bits)
assert L > 0 and sum(runs) == sum(run_bits)            # every 1 belongs to exactly one run
assert abs(sum(expected_run_count(L, n) for n in range(1, L + 1)) - expected_total_runs(L)) < 1e-9

n_max = max(6, max(runs, default=0))
lengths = np.arange(1, n_max + 1)
observed = np.array([runs.count(n) for n in lengths])
expected = np.array([expected_run_count(L, n) for n in lengths])
print(f"{source}: L = {L} bits, {sum(run_bits)} ones")
print(f"runs of 1s: {len(runs)};  a fair source gives (L + 1)/4 = {expected_total_runs(L):.2f} on average")
print(f"runs of length 5 or more: {sum(n >= 5 for n in runs)};  fair source: "
      f"{sum(expected_run_count(L, n) for n in range(5, L + 1)):.2f}\n")
print(pd.DataFrame({"run length n": lengths, "runs observed": observed, "fair source, expected": expected.round(2),
                    "fraction observed": (observed / max(len(runs), 1)).round(3),
                    "2^-n": (0.5 ** lengths).round(4)}).to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.bar(lengths, observed, width=0.6, color=DATA_COLOR, label="runs of 1s observed, by length n")
ax.plot(lengths, expected, "o", ms=8, color=PREDICTION_COLOR, markeredgecolor="white",
        markeredgewidth=2, label="fair source: expected number")
ax.set_xticks(lengths)
ax.set_xlabel("run length n")
ax.set_ylabel("number of runs")
ax.set_title(f"Run-length test on {L} bits: {source}", fontsize=9)
ax.grid(axis="y", color="0.9")
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

---
## Wrap-up — Back to the Bench

Every number in this notebook is one you produce at the bench:

- the manual's example (Part 1) is the bookkeeping of your worksheet: sift, test, and
  encrypt only if the test shows no error;
- the QBER and its SE at $Q = 1/4$ (Part 2) are the numbers of your W4 row;
- the class-pooled QBER (Part 3) answers the report question: is run 2 consistent with
  25 % within its SE, and is run 1 consistent with 0?
- the first-6-bits count (Part 4) answers "would a 6-bit test have caught Eve?";
- the run-length test (Part 5) is the playbox, run on your own bits.

**Replacing the SYNTHETIC data with your class's.** Each group copies
`lab9_worksheet_template.csv`, fills one row per pulse (bases `+` / `x`, bits `0` / `1`,
and `same` / `diff`, or `=` / `≠`, on the test bits) and writes its group name in every
row. Whoever collects the class data stacks all the groups' rows in one CSV with the same
columns. Upload that file next to this notebook, set `CLASS_FILE` and `MY_GROUP` in
Part 2, and rerun from Part 2 on: every table, band and fraction then describes your
class. The checks tied to the synthetic data (run 1 exactly 0, no flagged marks) switch
off; flagged marks are listed for you to correct in the CSV, never fixed silently. A few
run-1 errors on a real bench mean misalignment, which the ideal bench leaves out.

**Source.** Thorlabs, *EDU-QCRY1 Quantum Cryptography Demonstration Kit, User Manual*,
document MTN005660-D02, Rev D (2026-01-22): Chapter 6 (the worked example, transcribed per
pulse with attribution) and Chapter 10 (the randomness test). Further reading:
C. H. Bennett and G. Brassard, "Quantum cryptography: Public key distribution and coin
tossing", *Proc. IEEE Int. Conf. on Computers, Systems and Signal Processing* (1984),
reprinted in *Theor. Comput. Sci.* **560**, 7 (2014).

---
## Optional · The Kit's Labels as Qubit Circuits (Qiskit)

*Nothing above needs this part. The cell installs Qiskit if it is missing, and skips
itself with a one-line message if it cannot.*

**The question first:** *the kit calls bit 0 of the `x` basis "the −45 setting".
Written as a qubit, does that naming choice matter?*

**Why this step:** under the course convention $|0\rangle \equiv |H\rangle$ the kit's
labels are four qubit states: `+` bit 0 is $|0\rangle$, `+` bit 1 is $|1\rangle$, `x`
bit 1 is $|+\rangle$ ($+45°$) and `x` bit 0 is $|-\rangle$ ($-45°$). That last one is
the opposite of the common textbook rule "bit 0 is the $+1$ eigenstate": declared
deviation DEV-9.2, the kit's `x` labels. Bob's receiver turns the polarization and then
reads horizontal versus vertical, which is **Procedure 2** (rotate, then measure $Z$;
post-measurement state $|c\rangle$), not **Procedure 1** (a direct projection onto the
basis states); both give the same statistics. For the `x` basis the canonical rotation
is $U_X = H$, with $H|+\rangle = |0\rangle$ and $H|-\rangle = |1\rangle$. With the kit's
labels the measured bit $c$ is read the other way round, kit bit $= 1 - c$: the kit's
`x` receiver is $U_X = H$ with the labels swapped. Renaming both diagonals would change
no LED and no error count.

The cell prepares all $4 \times 2$ cases $(a, \alpha) \times \beta$ with the kit's labels
(`x` bit 0 as X then H, since $H|1\rangle = |-\rangle$), measures each on a seeded
simulator, and checks that $b = a$ whenever the bases match and that $b$ is a fair coin
when they do not. One pulse is one qubit, so Qiskit's bit order needs no conversion:
`qiskit_to_lab` is the identity on 1-bit strings.

In [ ]:
"""
OPTIONAL - the 8 kit cases (a, alpha) x beta as 1-qubit circuits (Qiskit).
Installs Qiskit if it is missing; if that fails, prints one line and skips.
"""
import importlib
import subprocess
import sys


def qiskit_available():
    try:
        importlib.import_module("qiskit")
        return True
    except ImportError:
        return False


if not qiskit_available():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qiskit"], check=False)
    importlib.invalidate_caches()

if not qiskit_available():
    print("Qiskit is not available: optional cell skipped (Parts 1-5 are unaffected).")
else:
    from qiskit import QuantumCircuit
    from qiskit.primitives import StatevectorSampler
    from qiskit.quantum_info import Statevector

    def qiskit_to_lab(bitstring):
        """Qiskit's bit order -> the lab's (a string reversal); the identity on 1 bit."""
        return bitstring[::-1]

    assert all(qiskit_to_lab(s) == s for s in ("0", "1"))

    def kit_circuit(a, alpha, beta):
        """Alice prepares (a, alpha) with the kit's labels; Bob measures in beta."""
        qc = QuantumCircuit(1)
        if (a, alpha) == (1, "+"):
            qc.x(0)             # |1>, the 90 setting
        elif (a, alpha) == (1, "x"):
            qc.h(0)             # |+>, the 45 setting
        elif (a, alpha) == (0, "x"):
            qc.x(0)
            qc.h(0)             # H|1> = |->, the -45 setting
        if beta == "x":
            qc.h(0)             # U_X = H, then measure Z (Procedure 2)
        qc.measure_all()
        return qc

    def kit_bit(c, beta):
        """Bob's kit bit from the measured bit c: + reads c, x reads 1 - c (labels swapped)."""
        return c if beta == "+" else 1 - c

    SHOTS = 1000
    cases = [(a, alpha, beta) for alpha in ("+", "x") for a in (0, 1) for beta in ("+", "x")]
    circuits = [kit_circuit(*case) for case in cases]
    results = StatevectorSampler(seed=SEED).run(circuits, shots=SHOTS).result()
    print(" a  alpha  beta   P(b = a), exact   b = a in the shots")
    for (a, alpha, beta), qc, result in zip(cases, circuits, results):
        p_c = Statevector(qc.remove_final_measurements(inplace=False)).probabilities()
        p_b_is_a = p_c[a] if beta == "+" else p_c[1 - a]
        counts = result.data.meas.get_counts()
        same = sum(v for c, v in counts.items() if kit_bit(int(qiskit_to_lab(c)), beta) == a)
        print(f" {a}    {alpha}      {beta}      {p_b_is_a:.3f}             {same}/{SHOTS}")
        if alpha == beta:
            assert abs(p_b_is_a - 1.0) < 1e-12 and same == SHOTS          # matched bases: b = a
        else:
            assert abs(p_b_is_a - 0.5) < 1e-12 and abs(same / SHOTS - 0.5) < 0.07   # a fair coin
    print("b = a on every matched-basis shot; mismatched bases give a fair coin.")